# Тестирование новых моделей
## Подготовка данных

In [1]:
array = []

array.append("Во время обучения модели machine learning, важно обеспечить достаточное количество данных для тренировки") 
array.append("Во время обучения модели машинного обучения, важно обеспечить достаточное количество данных для тренировки")
    
array.append("Использование API позволяет интегрировать различные сервисы в ваше приложение, упрощая процесс разработки") 
array.append("The use of an API allows you to integrate various services into your application, simplifying the development process")
    
array.append("Для анализа больших объемов данных рекомендуется использовать фреймворк Hadoop, который обеспечивает эффективную обработку данных") 
array.append("В процессе дебаггинга программы была обнаружена ошибка, связанная с неправильным использованием переменных")
    
array.append("При разработке user interface необходимо учитывать принципы user experience, чтобы сделать приложение максимально удобным для пользователя") 
array.append("При разработке пользовательского интерфейса необходимо учитывать принципы пользовательского опыта, чтобы сделать приложение максимально удобным для пользователя")
    
array.append("Летний отпуск всегда запоминается яркими моментами, проведёнными на берегу моря")
array.append("Летний отпуск всегда оставляет яркие впечатления, проведённые на пляже")
array.append("Летний отпуск всегда оставляет яркие впечатления, проведённые на пляже")
    
array.append("Вчера мы с друзьями проводили время на природе, наслаждаясь пикником и свежим воздухом.")
array.append("Вчера мы с друзьями провели время в помещении, скучая и наслаждаясь теплом")
    
array.append("The latest trends in fashion emphasize sustainability and eco-friendly materials")
array.append("The latest trends in fashion ignore sustainability and promote fast fashion")
    
array.append("В этом году зима выдалась особенно холодной, и снег покрыл все окрестности белым покрывалом")

array

['Во время обучения модели machine learning, важно обеспечить достаточное количество данных для тренировки',
 'Во время обучения модели машинного обучения, важно обеспечить достаточное количество данных для тренировки',
 'Использование API позволяет интегрировать различные сервисы в ваше приложение, упрощая процесс разработки',
 'The use of an API allows you to integrate various services into your application, simplifying the development process',
 'Для анализа больших объемов данных рекомендуется использовать фреймворк Hadoop, который обеспечивает эффективную обработку данных',
 'В процессе дебаггинга программы была обнаружена ошибка, связанная с неправильным использованием переменных',
 'При разработке user interface необходимо учитывать принципы user experience, чтобы сделать приложение максимально удобным для пользователя',
 'При разработке пользовательского интерфейса необходимо учитывать принципы пользовательского опыта, чтобы сделать приложение максимально удобным для пользовате

## Spacy
python -m spacy download ru_core_news_lg

In [2]:
import time
import torch, torch.nn.functional as F
import spacy

# 1. инициализация
start = time.time()
nlp = spacy.load("ru_core_news_lg")        

# 2. батч-обработка
texts = [str(x) for x in array]   # 15 строк
docs  = list(nlp.pipe(texts, batch_size=32))

# 3. тензоры + l2-нормализация
spacy_tensors = torch.stack([
    F.normalize(torch.from_numpy(doc.vector).float(), p=2, dim=0)
    for doc in docs
])

# 4. сохранение
#pd.DataFrame(spacy_vectors.numpy()).to_csv("spacy_vectors.csv", index=False)

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(spacy_tensors))
print(type(spacy_tensors[0]))
print(type(spacy_tensors[0][0]))

spacy_tensors         # посмотреть первый вектор

Elapsed: 1.048 s
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([[ 0.0471, -0.0748, -0.0610,  ...,  0.0665,  0.0973,  0.0512],
        [ 0.0189, -0.0891, -0.1064,  ...,  0.0217,  0.1077,  0.0567],
        [ 0.0193,  0.0275,  0.0398,  ...,  0.0553,  0.0604,  0.0291],
        ...,
        [-0.0394,  0.0593,  0.0911,  ...,  0.1502,  0.0110, -0.0526],
        [-0.0327,  0.0338,  0.0763,  ...,  0.1353, -0.0280, -0.0146],
        [-0.0047, -0.1529, -0.0808,  ..., -0.1064, -0.0031,  0.0686]])

## sentence-transformers/paraphrase-multilingual-mpnet-base-v2

In [3]:
import torch, gc, time
from sentence_transformers import SentenceTransformer

start = time.time()

mpnet_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-mpnet-base-v2')
mpnet_tensors = mpnet_model.encode(array, convert_to_tensor=True, normalize_embeddings=True)

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(mpnet_tensors))
print(type(mpnet_tensors[0]))
print(type(mpnet_tensors[0][0]))

del mpnet_model
gc.collect()   
torch.cuda.empty_cache()

mpnet_tensors

Elapsed: 3.431 s
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([[-0.0210,  0.0568, -0.0008,  ...,  0.0071, -0.0275, -0.0335],
        [-0.0191,  0.0611, -0.0013,  ...,  0.0122, -0.0316, -0.0372],
        [-0.0119,  0.0471, -0.0041,  ..., -0.0236,  0.0029, -0.0008],
        ...,
        [ 0.0525,  0.0380, -0.0045,  ..., -0.0058, -0.0288, -0.0250],
        [ 0.0138,  0.0647, -0.0034,  ...,  0.0049, -0.0106, -0.0121],
        [-0.0704,  0.0087, -0.0060,  ...,  0.0032, -0.0201, -0.0085]],
       device='cuda:0')

## intfloat/multilingual-e5-large-instruct 

In [4]:
import torch, gc, time
from sentence_transformers import SentenceTransformer

custom_array = []
for sentence in array:
    custom_array.append("passage: " + sentence)


print(custom_array[0])

start = time.time()

e5_model = SentenceTransformer("intfloat/multilingual-e5-large-instruct")
e5_tensors = e5_model.encode(custom_array, convert_to_tensor=True, normalize_embeddings=True)

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(e5_tensors))
print(type(e5_tensors[0]))
print(type(e5_tensors[0][0]))

del e5_model
gc.collect()   
torch.cuda.empty_cache()

e5_tensors

passage: Во время обучения модели machine learning, важно обеспечить достаточное количество данных для тренировки
Elapsed: 3.331 s
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([[ 0.0224,  0.0028, -0.0332,  ..., -0.0044, -0.0243,  0.0223],
        [ 0.0199,  0.0061, -0.0264,  ..., -0.0029, -0.0290,  0.0193],
        [ 0.0179,  0.0071,  0.0040,  ..., -0.0358, -0.0150,  0.0064],
        ...,
        [ 0.0351,  0.0312, -0.0321,  ...,  0.0072, -0.0395,  0.0002],
        [ 0.0133,  0.0260, -0.0459,  ..., -0.0140, -0.0577,  0.0097],
        [ 0.0262,  0.0196, -0.0255,  ..., -0.0047, -0.0388,  0.0361]],
       device='cuda:0')

## DeepPavlov/bert-base-multilingual-cased-sentence
вместо google-bert/bert-base-multilingual-cased

In [5]:
import torch, gc, time
from sentence_transformers import SentenceTransformer

start = time.time()

bert_model = SentenceTransformer("DeepPavlov/bert-base-multilingual-cased-sentence")
bert_tensors = bert_model.encode(array, convert_to_tensor=True, normalize_embeddings=True)

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(bert_tensors))
print(type(bert_tensors[0]))
print(type(bert_tensors[0][0]))

del bert_model
gc.collect()   
torch.cuda.empty_cache()

bert_tensors

# from transformers import AutoTokenizer, AutoModel
# import torch

# start = time.time()

# tok = AutoTokenizer.from_pretrained("DeepPavlov/bert-base-multilingual-cased-sentence")
# model = AutoModel.from_pretrained("DeepPavlov/bert-base-multilingual-cased-sentence")

# batch = tok(array, padding=True, truncation=True, return_tensors="pt")

# with torch.no_grad():
#     outputs = model(**batch)        # пример получения hidden-states показан на странице google-bert/bert-base-multilingual-cased

# # mean pooling (как описано для этой модели в metatext-описании)
# attention = batch["attention_mask"].unsqueeze(-1)      # [B, L, 1]
# masked = outputs.last_hidden_state * attention         # обнуляем padded позиции
# bert_tensors = masked.sum(1) / attention.sum(1)          # [B, 768]

# print(f"Elapsed: {time.time() - start:.3f} s")
# print(type(bert_tensors))
# print(type(bert_tensors[0]))
# bert_tensors

No sentence-transformers model found with name DeepPavlov/bert-base-multilingual-cased-sentence. Creating a new one with mean pooling.


Elapsed: 1.354 s
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([[ 0.0263, -0.0470,  0.0655,  ..., -0.0595, -0.0260,  0.0101],
        [ 0.0219, -0.0420,  0.0592,  ..., -0.0443, -0.0290,  0.0142],
        [ 0.0077, -0.0346,  0.0493,  ..., -0.0217, -0.0510, -0.0048],
        ...,
        [ 0.0169,  0.0622,  0.0568,  ...,  0.0151, -0.0283,  0.0099],
        [ 0.0346,  0.0280,  0.0886,  ...,  0.0156,  0.0075, -0.0063],
        [ 0.0196,  0.0048,  0.0900,  ..., -0.0422, -0.0334,  0.0024]],
       device='cuda:0')

## ai-forever/FRIDA
вместо ai-forever/sbert_large_nlu_ru 

In [6]:
import torch, gc, time
from sentence_transformers import SentenceTransformer

custom_array = []
for sentence in array:
    custom_array.append("paraphrase: " + sentence)

start = time.time()

frida_model = SentenceTransformer("ai-forever/FRIDA")          
frida_tensors = frida_model.encode(custom_array, convert_to_tensor=True, normalize_embeddings=True)  

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(frida_tensors))
print(type(frida_tensors[0]))
print(type(frida_tensors[0][0]))

del frida_model
gc.collect()   
torch.cuda.empty_cache()

frida_tensors

Elapsed: 2.188 s
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([[-0.0397, -0.0525, -0.0344,  ..., -0.0248,  0.0323,  0.0276],
        [-0.0399, -0.0455, -0.0377,  ..., -0.0258,  0.0314,  0.0259],
        [-0.0326, -0.0403, -0.0391,  ..., -0.0217,  0.0202,  0.0328],
        ...,
        [-0.0448, -0.0081,  0.0346,  ..., -0.0280,  0.0128,  0.0247],
        [-0.0225, -0.0011,  0.0104,  ..., -0.0285,  0.0166,  0.0215],
        [-0.0255, -0.0446, -0.0171,  ..., -0.0453,  0.0269,  0.0007]],
       device='cuda:0')

## jinaai/jina-embeddings-v3

In [7]:
import torch, gc, time
from sentence_transformers import SentenceTransformer

start = time.time()

jina_model = SentenceTransformer("jinaai/jina-embeddings-v3", trust_remote_code=True)

task = "retrieval.passage"
jina_tensors = jina_model.encode(
    array,
    task=task,
    prompt_name=task,
    convert_to_tensor=True,
    normalize_embeddings=True
)

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(jina_tensors))
print(type(jina_tensors[0]))
print(type(jina_tensors[0][0]))

del jina_model
gc.collect()   
torch.cuda.empty_cache()

jina_tensors

flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn is not installed. Using PyTorch native attention implementation.
flash_attn i

Elapsed: 4.254 s
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([[ 0.0111, -0.0679,  0.0369,  ...,  0.0255,  0.0038,  0.0327],
        [ 0.0128, -0.0679,  0.0454,  ...,  0.0305,  0.0010,  0.0371],
        [ 0.0287, -0.0400,  0.1279,  ..., -0.0413, -0.0106,  0.0189],
        ...,
        [ 0.0023,  0.0079, -0.0530,  ...,  0.0114,  0.0205, -0.0122],
        [-0.0099, -0.0251, -0.0815,  ...,  0.0033,  0.0005,  0.0084],
        [-0.1328,  0.0153, -0.0869,  ...,  0.0306,  0.0133,  0.0117]],
       device='cuda:0', dtype=torch.bfloat16)

## Linq-AI-Research/Linq-Embed-Mistral

In [ ]:
# import torch, gc, time
# from sentence_transformers import SentenceTransformer

# start = time.time()

# linq_model = SentenceTransformer("Linq-AI-Research/Linq-Embed-Mistral")
# linq_tensors = linq_model.encode(array, convert_to_tensor=True, normalize_embeddings=True)

# print(f"Elapsed: {time.time() - start:.3f} s")
# print(type(linq_tensors))
# print(type(linq_tensors[0]))
# print(type(linq_tensors[0][0]))

# del linq_model
# gc.collect()   
# torch.cuda.empty_cache()

# linq_tensors

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 224.00 MiB. GPU 0 has a total capacity of 15.55 GiB of which 237.69 MiB is free. Including non-PyTorch memory, this process has 14.78 GiB memory in use. Of the allocated memory 14.47 GiB is allocated by PyTorch, and 15.06 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## gte-Qwen2-1.5B-instruct

In [2]:
import torch, gc, time
from sentence_transformers import SentenceTransformer

start = time.time()

gte_model = SentenceTransformer("Alibaba-NLP/gte-Qwen2-1.5B-instruct", trust_remote_code=True)
gte_tensors = gte_model.encode(array, convert_to_tensor=True, normalize_embeddings=True)

print(f"Elapsed: {time.time() - start:.3f} s")
print(type(gte_tensors))
print(type(gte_tensors[0]))
print(type(gte_tensors[0][0]))

del gte_model
gc.collect()   
torch.cuda.empty_cache()

gte_tensors

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/284 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/146k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/55.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/901 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/370 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Elapsed: 118.272 s
<class 'torch.Tensor'>
<class 'torch.Tensor'>
<class 'torch.Tensor'>


tensor([[-0.0345,  0.0362,  0.0419,  ..., -0.0359,  0.0152,  0.0142],
        [-0.0299,  0.0300,  0.0431,  ..., -0.0320,  0.0075,  0.0145],
        [ 0.0216,  0.0809, -0.0146,  ...,  0.0040,  0.0460, -0.0087],
        ...,
        [ 0.0353, -0.0063,  0.0143,  ..., -0.0226, -0.0097, -0.0332],
        [ 0.0116,  0.0129, -0.0159,  ..., -0.0181,  0.0083, -0.0220],
        [ 0.0099, -0.0401, -0.0512,  ..., -0.0667,  0.0105, -0.0243]],
       device='cuda:0')

In [ ]:
print(spacy_tensors.shape)
print(mpnet_tensors.shape)
print(e5_tensors.shape)
print(bert_tensors.shape)
print(frida_tensors.shape)
print(jina_tensors.shape)
# print(linq_tensors.shape)
print(gte_tensors.shape)

torch.Size([16, 300])
torch.Size([16, 768])
torch.Size([16, 1024])
torch.Size([16, 768])
torch.Size([16, 1536])
torch.Size([16, 1024])


## Оценка результатов (Косинусное сходство)

In [ ]:
import torch.nn.functional as F

def compute_sentence_similarities(embeddings):
    """
    Вычисляет cosine similarity для разных кейсов:
      1) перевод слова с EN→RU
      2) одинаковые предложения на RU и EN
      3) разные предложения в общем контексте
      4) перевод словосочетания с EN→RU
      5) разные предложения в схожем контексте (кросс-кейс)
      6) синонимичные предложения
      7) антонимичные предложения на русском
      8) антонимичные предложения на английском
      9) вообще разные предложения без общих контекстов
    """
    word_translation_similarity = F.cosine_similarity(
        embeddings[0], embeddings[1], dim=0
    )  # два слова заменены с английского на русский

    identical_sentence_ru_en_similarity = F.cosine_similarity(
        embeddings[2], embeddings[3], dim=0
    )  # два одинаковых предложения: русское ↔ английское

    context_overlap_similarity = F.cosine_similarity(
        embeddings[4], embeddings[5], dim=0
    )  # два разных предложения, но в общем контексте

    phrase_translation_similarity = F.cosine_similarity(
        embeddings[6], embeddings[7], dim=0
    )  # два словосочетания перевода EN→RU

    cross_context_similarity = F.cosine_similarity(
        embeddings[0], embeddings[7], dim=0
    )  # два предложения в схожем, но не идентичном контексте

    synonym_sentences_similarity = F.cosine_similarity(
        embeddings[8], embeddings[9], dim=0
    )  # два синонимичных предложения

    antonym_ru_sentences_similarity = F.cosine_similarity(
        embeddings[10], embeddings[11], dim=0
    )  # два антонимичных предложения на русском

    antonym_en_sentences_similarity = F.cosine_similarity(
        embeddings[12], embeddings[13], dim=0
    )  # два антонимичных предложения на английском

    unrelated_sentences_similarity = F.cosine_similarity(
        embeddings[14], embeddings[1], dim=0
    )  # два разных предложения без общего контекста

    return [
        word_translation_similarity,
        identical_sentence_ru_en_similarity,
        context_overlap_similarity,
        phrase_translation_similarity,
        cross_context_similarity,
        synonym_sentences_similarity,
        antonym_ru_sentences_similarity,
        antonym_en_sentences_similarity,
        unrelated_sentences_similarity,
    ]

